# SN-01 — Sirenisation, phase 1 : SIREN exact

Pour chaque personne morale portant un SIREN, on cherche l'unite legale
correspondante dans SIRENE et on score la paire obtenue.

La mise en correspondance est desormais une jointure et non un parcours ligne a
ligne ; seul le scoring reste sequentiel, et uniquement sur les PM effectivement
appariees.

Sorties : le fichier Excel de phase 1, et un parquet des PM validees qui evite
aux notebooks suivants de relire l'Excel.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.sirenisation import matching_direct_siren, COLS_UL_EXPORT
from src.excel_export import export_phase1_excel, LABELS
from src.display import afficher_tableau, afficher_synthese
from config.settings import (
    FINESS_PM_CLEAN, SIRENE_UL_CLEAN, SN_PHASE1, SN_VALIDES_P1,
    RESULTS_SN_DIR, PROCESSED_DIR,
)

RESULTS_SN_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
VALIDES = {"VALIDE_FORT", "VALIDE"}

In [2]:
df_pm = pd.read_parquet(FINESS_PM_CLEAN)

COLS_UL = ["siren", "denomination_norm_ul", "sigle_norm_ul",
           "code_commune_norm_ul", "numero_voie_norm_ul",
           "libelle_voie_complet_ul", "dept_ul"] + COLS_UL_EXPORT
df_ul = pd.read_parquet(SIRENE_UL_CLEAN)
df_ul = df_ul[[c for c in dict.fromkeys(COLS_UL) if c in df_ul.columns]]

print(f"PM FINESS : {len(df_pm):,}")
print(f"UL SIRENE : {len(df_ul):,}")

PM FINESS : 56,193
UL SIRENE : 15,111,205


In [3]:
df_resultats = matching_direct_siren(df_pm, df_ul)
print()
print(df_resultats["statut"].value_counts())

PM traitées      : 56,193
   sans SIREN    : 2,090
   SIREN inconnu : 5,604
   à scorer      : 48,499


Phase 1 sirenisation:   0%|          | 0/48499 [00:00<?, ?it/s]


statut
VALIDE_FORT      23085
VALIDE           15700
DOUTEUX           6870
SIREN_INCONNU     5604
REJETE            2844
SANS_SIREN        2090
Name: count, dtype: int64


In [4]:
afficher_tableau(
    df_resultats[df_resultats["statut"].isin(VALIDES)],
    "Apercu validees phase 1", max_lignes=9,
    colonnes=["NB_PmSmsseId", "TX_NumFinessPm", "siren_norm_pm", "TX_DenominationPm",
              "denominationUniteLegale", "nom_ul_retenu",
              "score_nom", "score_adresse", "score_global", "statut"])

NB_PmSmsseId,TX_NumFinessPm,siren_norm_pm,TX_DenominationPm,denominationUniteLegale,nom_ul_retenu,score_nom,score_adresse,score_global,statut
2,010000156,772201489,CLINIQUE CONVERT,CLINIQUE CONVERT,CLINIQUE CONVERT,100.000000,100.000000,100.000000,VALIDE_FORT
9,010000255,779309194,ASS.ACCUEIL FORMA INSERT PERSON SOURD,POUR L ACCUEIL LA FORMATION ET L INSERTION DES PERSONNES SOURDES,ACCUEIL FORMATION INSERTION PERSONNES SOURDES,67.740000,100.000000,87.100000,VALIDE_FORT
17,010000339,260100011,RESIDENCE FONTELUNE,RESIDENCE FONTELUNE,RESIDENCE FONTELUNE,100.000000,100.000000,100.000000,VALIDE_FORT
18,010000347,260100029,LA RESIDENCE D'URFE,RESIDENCE D'URFE,RESIDENCE URFE,100.000000,100.000000,100.000000,VALIDE_FORT
19,010000354,260100052,EHPAD L'ALBIZIA,MAISON DE RETRAITE-EHPAD,MAISON RETRAITE EHPAD,32.590000,70.000000,55.040000,VALIDE
20,010000362,260100078,EHPAD FONDATION COSTAZ CHAMPAGNE,EHPAD FONDATION COSTAZ CHAMPAGNE,EHPAD FONDATION COSTAZ CHAMPAGNE,100.000000,100.000000,100.000000,VALIDE_FORT
21,010000388,260100094,MAISON DE RETRAITE DE COLIGNY,EHPAD DE COLIGNY RESIDENCE LA JONQUILLERE,EHPAD COLIGNY RESIDENCE JONQUILLERE,29.080000,58.440000,46.700000,VALIDE
22,010000396,260100128,MAISON DE RETRAITE DE LAGNIEU,MAISON DE RETRAITE DE LAGNIEU,MAISON RETRAITE LAGNIEU,100.000000,88.920000,93.350000,VALIDE_FORT
23,010000404,260100144,MAISON DE RETRAITE DE MONTLUEL,MAISON DE RETRAITE DE MONTLUEL,MAISON RETRAITE MONTLUEL,100.000000,55.460000,73.280000,VALIDE


In [5]:
COLS_COMPLET = [
    "ID_PmSmsse", "NB_PmSmsseId", "TX_NumFinessPm", "TX_Siren",
    "TX_CategorieEntiteGeographiqueExercice", "TX_DenominationPm",
    "TX_EtatActif", "cdcommune_norm_pm", "adresse_complete_pm",
    "siren_ul", "denominationUniteLegale", "sigleUniteLegale", "nom_ul_retenu",
    "adresse_siege_complete_ul", "codeCommuneEtablissement",
    "categorieJuridiqueUniteLegale", "activitePrincipaleUniteLegale",
    "dateCreationUniteLegale",
    "score_nom", "score_adresse", "score_global",
]
COLS_INFO = [
    "ID_PmSmsse", "NB_PmSmsseId", "TX_NumFinessPm", "TX_Siren",
    "TX_DenominationPm", "cdcommune_norm_pm", "adresse_complete_pm",
]

compteurs = export_phase1_excel(
    df_resultats, SN_PHASE1, COLS_COMPLET, COLS_INFO,
    statuts_score=["VALIDE_FORT", "VALIDE", "DOUTEUX", "REJETE"],
    statuts_info=["SANS_SIREN", "SIREN_INCONNU"])

afficher_synthese({LABELS[s]: n for s, n in compteurs.items()},
                  "Synthese sirenisation phase 1")
print(f"\nFichier : {SN_PHASE1}")

Statut,Nb,% du total
Valide_fort,"23,085",41.1%
Valide,"15,700",27.9%
Douteux,"6,870",12.2%
Rejeté,"2,844",5.1%
Sans_SIREN,"2,090",3.7%
SIREN_inconnu,"5,604",10.0%
TOTAL,"56,193",100.0%



Fichier : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/results/sirenisation/sirenisation_phase1.xlsx


In [6]:
valides = df_resultats[df_resultats["statut"].isin(VALIDES)][
    ["NB_PmSmsseId", "siren_norm_pm", "statut"]].copy()
valides.to_parquet(SN_VALIDES_P1, index=False)
print(f"{len(valides):,} PM validees -> {SN_VALIDES_P1}")

38,785 PM validees -> /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/processed/sirenisation_valides_p1.parquet
